In [ ]:
!pip install -U "transformers" "bitsandbytes" "trl" "datasets" "evaluate" "matplotlib-venn" "peft" "accelerate"

In [ ]:
import torch
import multiprocessing

if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.get_device_name(0)}")
    print(f"Device count: {torch.cuda.device_count()}")
else:
    print("CUDA is not available. Please check your runtime settings.")

# Optimization: Set high precision for matmul if using Tensor Cores
torch.set_float32_matmul_precision('high')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset("Blpeng/nsmc")

# Filter empty documents
dataset = dataset.filter(lambda x: x["document"] is not None and x["document"].strip() != "")

# Tokenizer
model_id = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess(batch):
    return tokenizer(
        batch["document"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Map preprocessing
tokenized = dataset.map(preprocess, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Split
split_datasets = tokenized['train'].train_test_split(test_size=0.2)
train_ds = split_datasets['train']
eval_ds = split_datasets['test']

print(f"Training size: {len(train_ds)}")
print(f"Validation size: {len(eval_ds)}")

In [ ]:
from transformers import AutoModelForSequenceClassification
import evaluate
import numpy as np

# Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)

# Metrics
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [10]:
from transformers import TrainingArguments, Trainer

# OPTIMIZED SETTINGS FOR RTX 5070
batch_size = 64
num_workers = 4

training_args = TrainingArguments(
    output_dir="nsmc-lora-klue-bert-optimized",
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=20,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    dataloader_num_workers=num_workers,
    fp16=True,
    report_to="none",
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training with optimized settings...")
trainer.train()

C:\Users\daboi\AppData\Local\Temp\ipykernel_9308\1263093490.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training with optimized settings...


Step,Training Loss,Validation Loss,Accuracy
200,0.134700,0.377074,0.877229
400,0.117500,0.402208,0.871262
600,0.283800,0.281505,0.885996
800,0.252100,0.285263,0.886296
1000,0.288100,0.256717,0.893696
1200,0.290300,0.291264,0.882163
1400,0.231700,0.260063,0.894396
1600,0.244400,0.245588,0.900097
1800,0.231600,0.264534,0.893830
2000,0.185000,0.266645,0.901097


TrainOutput(global_step=5625, training_loss=0.1734495127836863, metrics={'train_runtime': 1929.5576, 'train_samples_per_second': 186.565, 'train_steps_per_second': 2.915, 'total_flos': 2.367920564923392e+16, 'train_loss': 0.1734495127836863, 'epoch': 3.0})

In [11]:
eval_results = trainer.evaluate()

print(f"최종 정확도(Accuracy): {eval_results['eval_accuracy']:.4f}")

최종 정확도(Accuracy): 0.9047


In [12]:
save_dir = "./klue-bert-best"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"모델이 '{save_dir}' 폴더에 저장")

모델이 './klue-bert-best' 폴더에 저장


In [23]:
import torch

text = "I feel melancholy"

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()

labels = ['negative', 'positive']

print(f"입력 문장: {text}")
print(f"예측 감정: {labels[predicted_class_id]}")

입력 문장: I feel melancholy
예측 감정: positive
